# MiniMind-O 02 — 多模态交互推理、批量评估与零样本音色克隆

本 Notebook 提供完整的笔记本内推理体验：
- 文本 / 语音 / 图像 / 混合模态输入，实时流式文本输出 + 24kHz 语音内联播放
- Colab 浏览器麦克风实时录音提问
- 使用内置音色、未见音色或任意自定义参考音频进行 Zero-Shot 音色克隆

In [ ]:
#@title 1. 环境初始化与依赖安装 (Colab / 本地 Jupyter 通用)
import os
import shutil
import subprocess
import sys

REPO_URL = "https://github.com/jingyaogong/minimind-o.git"  #@param {type:"string"}
BRANCH = ""  #@param {type:"string"} # 留空克隆默认分支，或填入特定分支名
INSTALL_DEPS = True  #@param {type:"boolean"}

def _find_repo_root():
    for cand in [os.getcwd(), os.path.dirname(os.getcwd()), "/content/minimind-o"]:
        if os.path.exists(os.path.join(cand, "notebook_utils.py")) and os.path.exists(os.path.join(cand, "model", "model_omni.py")):
            return os.path.abspath(cand)
    return None

root = _find_repo_root()
if root is None:
    target = "/content/minimind-o" if "google.colab" in sys.modules else os.path.abspath("./minimind-o")
    cmd = ["git", "clone", "--depth", "1"]
    if BRANCH.strip():
        cmd += ["-b", BRANCH.strip()]
    cmd += [REPO_URL, target]
    print("Cloning repository:", " ".join(cmd))
    subprocess.run(cmd, check=True)
    root = target

os.chdir(root)
if root not in sys.path:
    sys.path.insert(0, root)

if INSTALL_DEPS:
    req_file = "requirements_colab.txt" if os.path.exists("requirements_colab.txt") else "requirements.txt"
    print(f"Installing dependencies from {req_file} ...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", req_file], check=True)
    if shutil.which("ffmpeg") is None and "google.colab" in sys.modules:
        subprocess.run(["apt-get", "update", "-qq"], check=False)
        subprocess.run(["apt-get", "install", "-y", "-qq", "ffmpeg"], check=False)

from notebook_utils import check_environment
env_info = check_environment()

In [ ]:
#@title 2. 下载编码器与推理权重
SOURCE = "huggingface"  #@param ["huggingface", "modelscope"]
MODEL_VARIANT = "minimind-3o"  #@param ["minimind-3o", "minimind-3o-moe"]
LOAD_FORMAT = "hf_dir"  #@param ["hf_dir", "pytorch_pth"]

from notebook_utils import download_encoders, download_hf_model, download_pytorch_weights

download_encoders(source=SOURCE, include_campplus=True)
if LOAD_FORMAT == "hf_dir":
    download_hf_model(model_name=MODEL_VARIANT, source=SOURCE, target_dir=os.path.join(".", MODEL_VARIANT))
else:
    suffix = "_moe" if MODEL_VARIANT == "minimind-3o-moe" else ""
    download_pytorch_weights(filenames=[f"sft_omni_768{suffix}.pth"], source=SOURCE, save_dir="./out")

In [ ]:
#@title 3. 初始化模型会话
OPEN_THINKING = False  #@param {type:"boolean"}

from eval_omni import EvalConfig
from notebook_utils import OmniNotebookSession

use_moe = (MODEL_VARIANT == "minimind-3o-moe")
cfg = EvalConfig(
    load_from=os.path.join(".", MODEL_VARIANT) if LOAD_FORMAT == "hf_dir" else "model",
    save_dir="./out",
    weight="sft_omni",
    use_moe=int(use_moe),
    open_thinking=int(OPEN_THINKING),
    device="cuda" if env_info["cuda_available"] else "cpu",
)
session = OmniNotebookSession(cfg)
print("已加载音色列表:", session.list_voices())

In [ ]:
#@title 4. 交互式多模态对话（支持文本 / 图像 / 音频 / 多轮历史）
TEXT_PROMPT = "Tell me an interesting fact about space."  #@param {type:"string"}
IMAGE_PATH = ""  #@param {type:"string"} # 可填 ./dataset/eval_omni/image-01-orange-cat-moon-desk.jpg
AUDIO_PATH = ""  #@param {type:"string"} # 可填 ./dataset/eval_omni/audio-en-01_what_do_you_usually_like_to_eat.mp3
VOICE_NAME = "default"  #@param ["default", "dylan", "eric", "serena", "uncle_fu", "vivian", "arthur", "chelsie", "cherry", "ethan", "jennifer", "momo", "moon"]
TEMPERATURE = 0.7  #@param {type:"number"}

res = session.chat(
    text=TEXT_PROMPT,
    image=IMAGE_PATH.strip() or None,
    audio_path=AUDIO_PATH.strip() or None,
    voice_name=VOICE_NAME,
    temperature=TEMPERATURE,
)

In [ ]:
#@title 5. (Colab) 浏览器麦克风录音并直接向 MiniMind-O 提问
RECORD_SECONDS = 4  #@param {type:"integer"}
VOICE_NAME_MIC = "default"  #@param ["default", "dylan", "serena", "uncle_fu", "arthur"]

from notebook_utils import is_colab, record_audio_colab

if is_colab():
    mic_wav = record_audio_colab("mic_query.wav", seconds=RECORD_SECONDS)
    session.chat(audio_path=mic_wav, voice_name=VOICE_NAME_MIC)
else:
    print("当前非 Google Colab 环境，请直接传入本地录音文件 path 给 session.chat(audio_path=...)。")

In [ ]:
#@title 6. 自定义参考音频音色克隆 (Mimi + CAM++)
REF_AUDIO_PATH = "./dataset/eval_omni/audio-zh-01_你平时喜欢吃什么.mp3"  #@param {type:"string"}
NEW_VOICE_NAME = "my_custom_voice"  #@param {type:"string"}
TEST_CLONE_TEXT = "你好，这是使用自定义参考音频克隆的新音色测试。"  #@param {type:"string"}

if os.path.exists(REF_AUDIO_PATH):
    cloned = session.clone_voice_from_audio(REF_AUDIO_PATH, voice_name=NEW_VOICE_NAME, save=True)
    session.chat(text=TEST_CLONE_TEXT, voice_name=cloned)
else:
    print("未找到参考音频:", REF_AUDIO_PATH)

In [ ]:
#@title 7. 批量评估套件（文 / 音 / 图 / 混合模态）
EVAL_MODES = "0,2,4,5"  #@param {type:"string"}
PROMPT_LANG = 1  #@param {type:"integer"} # 0=英文, 1=中文, 2=中英文
MAX_ITEMS = 1  #@param {type:"integer"}

_ = session.run_eval_suite(mode=EVAL_MODES, prompt_lang=PROMPT_LANG, max_items=MAX_ITEMS)